In [2]:
import pandas as pd
import numpy as np

# 1. Load data
orders = pd.read_csv('/content/drive/MyDrive/orders.csv')
support = pd.read_csv('/content/drive/MyDrive/support_tickets.csv')
customers = pd.read_csv('/content/drive/MyDrive/customers.csv')
interventions = pd.read_csv('/content/drive/MyDrive/intervention_history.csv')

# Ensure correct date formats
orders['order_date'] = pd.to_datetime(orders['order_date'])
support['ticket_date'] = pd.to_datetime(support['ticket_date'])

SNAPSHOT_DATE = pd.to_datetime('2025-09-30')

# 2. Strict Leakage Prevention: Filter out post-snapshot orders
pre_snapshot_orders = orders[
    (orders['order_date'] <= SNAPSHOT_DATE) &
    (~orders['order_id'].str.contains('_DUP'))
].copy()

# 3. Calculate Base RFM metrics
# Recency: Days between customer's last order and snapshot date
# Frequency: Count of unique orders pre-snapshot
# Monetary: Sum of gross_amount minus the discount applied
pre_snapshot_orders['net_amount'] = pre_snapshot_orders['gross_amount'] * (1 - pre_snapshot_orders['discount_pct'])

rfm_base = pre_snapshot_orders.groupby('customer_id').agg(
    last_order_date=('order_date', 'max'),
    frequency=('order_id', 'nunique'),
    monetary=('net_amount', 'sum')
).reset_index()

rfm_base['recency'] = (SNAPSHOT_DATE - rfm_base['last_order_date']).dt.days

# 4. Integrate Non-RFM Signals (Support Complaints & Return Rates)
# Signal A: Return Rate per customer
returns = pre_snapshot_orders.groupby('customer_id')['returned'].mean().reset_index(name='return_rate')

# Signal B: Support ticket counts within the last 90 days prior to snapshot
support_90d = support[support['ticket_date'] >= (SNAPSHOT_DATE - pd.Timedelta(days=90))]
ticket_counts = support_90d.groupby('customer_id').size().reset_index(name='recent_tickets')

# Merge metrics back to our primary 2,400 customer universe
df_segments = customers[['customer_id']].merge(rfm_base, on='customer_id', how='left')
df_segments = df_segments.merge(returns, on='customer_id', how='left').fillna({'return_rate': 0.0})
df_segments = df_segments.merge(ticket_counts, on='customer_id', how='left').fillna({'recent_tickets': 0})

# Handle zero-order customers by imputing worst-case baseline metrics
df_segments['recency'] = df_segments['recency'].fillna(180)
df_segments['frequency'] = df_segments['frequency'].fillna(0)
df_segments['monetary'] = df_segments['monetary'].fillna(0.0)

# 5. Segment Scoring Logic (Quantile Brackets 1-5)
# Note: Higher R score means MORE recent (lower recency days)
df_segments['R_score'] = pd.qcut(df_segments['recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)
df_segments['F_score'] = pd.qcut(df_segments['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
df_segments['M_score'] = pd.qcut(df_segments['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

# 6. Apply Segment Mapping Profile Rule
def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    ret_rate = row['return_rate']
    tickets = row['recent_tickets']

    # Tier 1: High Spenders & Highly Active
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    # Tier 2: Solid Frequency but slipping away in Recency
    elif r <= 2 and f >= 3:
        return 'At-Risk Customers'
    # Tier 3: Zero or near-zero transactional signs
    elif f <= 1 or r >= 5:
        return 'Dormant Customers'
    # Tier 4: High return rates or compounding support issues
    elif ret_rate > 0.4 or tickets >= 2:
        return 'Frustrated / High-Complaint'
    # Default Tier: Average fallback profiles
    else:
        return 'Loyal / Core Buyers'

df_segments['segment_name'] = df_segments.apply(assign_segment, axis=1)

# Save requested metrics file to repository root
output_cols = ['customer_id', 'segment_name', 'recency', 'frequency', 'monetary', 'return_rate', 'recent_tickets']
df_segments[output_cols].to_csv('segments.csv', index=False)
print("Successfully generated and saved segments.csv with automated logic labels.")

Successfully generated and saved segments.csv with automated logic labels.
